# Tydzień 7: Uczenie nienadzorowane – grupowanie (*Clustering*) algorytmem K-Means 🧩

| | |
|---|---|
| **Rozdział książki** | 4. *Clustering* |
| **Czas zajęć** | 90 minut |
| **Dane** | syntetyczny (sztuczny) zbiór wygenerowany funkcją `make_blobs()` z biblioteki scikit-learn |
| **Nowe funkcje PyCaret** | moduł `pycaret.clustering`: `create_model('kmeans')`, `plot_model(..., 'elbow')`, `assign_model()` |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. rozumieć, czym jest **uczenie nienadzorowane** i **grupowanie** (*clustering*),
2. wiedzieć krok po kroku, jak działa algorytm **K-Means** (metoda k-średnich),
3. umieć dobrać liczbę grup **metodą łokcia** (*Elbow Method*) i ocenić grupowanie **współczynnikiem sylwetki** (*Silhouette*),
4. umieć zwizualizować grupy na wykresie **PCA** i przypisać każdej obserwacji numer grupy.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–20 min | 📚 Teoria: K-Means, metoda łokcia, sylwetka, PCA |
| 20–65 min | 👣 Krok po kroku |
| 65–85 min | 🧑‍💻 Ćwiczenie: znajdź grupy w nowych danych – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Uczenie nienadzorowane – bez „klucza odpowiedzi” 🗝️❌
Do tej pory każdy przykład miał poprawną odpowiedź (koszt ubezpieczenia, gatunek kwiatu). Teraz **nie mamy żadnych etykiet**.
Zadanie: **znaleźć grupy podobnych obserwacji**. To jak sortowanie pudła pomieszanych skarpetek 🧦 – nikt nie mówi, które
pasują do siebie, ale widać, które są podobne.

Przykład z życia: **segmentacja klientów** (*market segmentation*) – sklep dzieli klientów na grupy (np. „oszczędni”,
„rodzinni”, „lubiący nowości”) i przygotowuje dla każdej grupy inną reklamę.

## Algorytm K-Means – „gdzie postawić pizzerie?” 🍕
Chcemy otworzyć w mieście **K pizzerii** tak, żeby każdy klient miał blisko do najbliższej z nich.
1. **Start:** stawiamy K pizzerii w losowych miejscach (to tzw. **centroidy**, środki grup).
2. **Przypisanie:** każdy klient chodzi do **najbliższej** pizzerii → powstaje K grup klientów.
3. **Przesunięcie:** każdą pizzerię przenosimy na **środek** jej klientów (średnia ich położeń – stąd nazwa „k-średnich”).
4. **Powtarzamy** kroki 2–3, aż pizzerie przestaną się przesuwać.

Algorytm stara się, by klienci w każdej grupie byli **jak najbliżej siebie** (mała zmienność wewnątrz grup):

$$\text{minimalizuj} \sum_{k=1}^{K} W(C_k), \quad W(C_k) = \text{„rozrzut” punktów wewnątrz grupy } C_k$$

⚠️ **Haczyk:** liczbę grup **K musimy podać sami**. Skąd ją wziąć?

## Metoda łokcia (*Elbow Method*) 💪
Trenujemy K-Means dla K = 2, 3, 4, … i dla każdego K liczymy, jak bardzo punkty są „rozrzucone” wokół swoich środków
(tzw. *distortion score*). Im więcej pizzerii, tym bliżej do każdej – ale **od pewnego momentu kolejne pizzerie pomagają już niewiele**.
Na wykresie krzywa gwałtownie spada, a potem się „wypłaszcza” – miejsce zgięcia przypomina **łokieć** i wskazuje dobre K.

## Współczynnik sylwetki (*Silhouette Coefficient*) 👤
Dla każdego punktu pytamy: *„czy jestem bliżej swojej grupy niż najbliższej obcej grupy?”*

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}, \qquad -1 \le s(i) \le 1$$

- $a(i)$ – średnia odległość punktu od członków **własnej** grupy,
- $b(i)$ – średnia odległość od członków **najbliższej innej** grupy.

| Wartość | Znaczenie |
|---|---|
| blisko **1** | punkt świetnie pasuje do swojej grupy 👍 |
| około **0** | punkt leży na granicy dwóch grup (grupy nachodzą na siebie) |
| **ujemna** | punkt chyba trafił do złej grupy 👎 |

## PCA – „cień” danych na ścianie 🔦
Nasze dane mają 5 kolumn (5 wymiarów), a ekran tylko 2. **PCA** (*Principal Component Analysis*, analiza głównych składowych)
to jak oświetlenie przedmiotu latarką tak, by jego **cień** na ścianie był jak najbardziej „informacyjny”.
Zamieniamy 5 kolumn na 2 nowe (`PCA1`, `PCA2`), tracąc jak najmniej informacji. Technika ta to **redukcja wymiarowości**.

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importy

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Biblioteki do tabel i wykresów
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Moduł GRUPOWANIA PyCaret oraz funkcja make_blobs, która generuje sztuczne dane z wyraźnymi grupami
from pycaret.clustering import *
from sklearn.datasets import make_blobs

### Krok 2: Generujemy sztuczny zbiór danych 🧪
Zamiast pobierać dane, **wygenerujemy** je sami. Zaleta: wiemy, że są w nich **dokładnie 3 grupy**, więc łatwo sprawdzimy,
czy algorytm je znajdzie. (Prawdziwe dane rzadko są tak „grzeczne”, ale na start to idealny trening.)

In [ ]:
# 1000 punktów, 5 cech, 3 grupy (centers=3); cluster_std – jak bardzo "rozmyte" są grupy; random_state – powtarzalność
X, true_labels = make_blobs(n_samples=1000, n_features=5, centers=3, cluster_std=1, random_state=20)

In [ ]:
# Zamieniamy tablicę liczb na tabelę pandas z nazwami kolumn
cols = ['column1', 'column2', 'column3', 'column4', 'column5']
data = pd.DataFrame(X, columns=cols)

In [ ]:
# Pierwsze wiersze – zwykłe liczby, ŻADNYCH etykiet grup (true_labels schowaliśmy na koniec zajęć 🤫)
data.head()

### Krok 3: Szybka EDA 🔍

In [ ]:
# Histogramy wszystkich kolumn; grid=False wyłącza siatkę
data.hist(bins=30, figsize=(12, 10), grid=False)
plt.show()

🔎 Histogramy mają **dwa lub więcej „garbów”** (rozkład dwu- lub wielomodalny, *bimodal / multimodal*).
To typowy znak, że w danych ukrywa się kilka grup o różnych cechach.

In [ ]:
# Mapa korelacji między kolumnami
sns.heatmap(data.corr().round(2), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

In [ ]:
# Wykresy rozrzutu dla par kolumn (pierwsze 4 kolumny, małe kropki s=5 – szybciej się rysuje)
sns.pairplot(data, vars=cols[:4], plot_kws={'s': 5})
plt.show()

🔎 Na wykresach rozrzutu gołym okiem widać **wyraźnie oddzielone „chmurki”** punktów – to nasze grupy.

### Krok 4: Inicjalizacja środowiska – `setup()`
W grupowaniu **nie ma zmiennej celu**, więc nie podajemy parametru `target`!

In [ ]:
# Domyślne ustawienia w zupełności wystarczą; session_id – powtarzalność wyników
cluster = setup(data, session_id=7652)

### Krok 5: Pierwszy model K-Means 🍕

In [ ]:
# 'kmeans' = algorytm K-Means. Domyślnie PyCaret tworzy 4 grupy (num_clusters=4)
model = create_model('kmeans')

📖 **Jak czytać tabelę?** Skupiamy się na **Silhouette** (sylwetka): ok. **0,57** – nieźle, ale da się lepiej.
Kolumny `Homogeneity`, `Rand Index`, `Completeness` są równe 0, bo wymagają prawdziwych etykiet grup – a w uczeniu
nienadzorowanym zwykle ich nie mamy.

### Krok 6: Metoda łokcia – ile naprawdę jest grup? 💪

In [ ]:
# Wykres łokcia: oś X = liczba grup K, oś Y = "rozrzut" punktów (distortion score); przerywana linia = wykryty łokieć
plot_model(model, plot='elbow')

🔎 Krzywa gwałtownie spada do **K = 3**, a potem prawie się nie zmienia – łokieć jest przy **K = 3**. Budujemy model ponownie:

In [ ]:
# K-Means z 3 grupami
model = create_model('kmeans', num_clusters=3)

🔎 Sylwetka wzrosła z ok. 0,57 do ok. **0,80** – grupy są teraz dużo lepiej dopasowane. 🎉

### Krok 7: Wizualizacja grup – wykres PCA 🔦

In [ ]:
# Wykres 2D po redukcji wymiarów metodą PCA (interaktywny: możesz najechać myszką na punkt i powiększać)
plot_model(model, plot='cluster')

🔎 Trzy grupy są **wyraźnie rozdzielone** – każda „chmurka” ma swój kolor.

In [ ]:
# Wykres sylwetki: każdy "pasek" to jeden punkt; czerwona przerywana linia = średnia sylwetka
plot_model(model, plot='silhouette')

### Krok 8: Przypisanie grup do danych – `assign_model()` 🏷️

In [ ]:
# assign_model dodaje do danych kolumnę 'Cluster' z numerem grupy dla każdego wiersza
results = assign_model(model)
results.head(10)

In [ ]:
# Ile punktów trafiło do każdej grupy?
results['Cluster'].value_counts()

In [ ]:
# Średnie wartości cech w każdej grupie – "portret" każdej grupy (tak opisujemy segmenty klientów!)
results.groupby('Cluster').mean()

### Krok 9: Sprawdzian – czy K-Means znalazł prawdziwe grupy? 🕵️
Teraz wyciągamy schowane `true_labels`. Tabela krzyżowa (*crosstab*) pokazuje, ile punktów z każdej prawdziwej grupy
trafiło do każdej znalezionej grupy.

In [ ]:
# Wiersze = grupy znalezione przez K-Means, kolumny = prawdziwe grupy z make_blobs
pd.crosstab(results['Cluster'], true_labels)

🔎 W każdym wierszu tylko **jedna** liczba jest różna od zera – każda znaleziona grupa odpowiada dokładnie jednej prawdziwej grupie.
Numery mogą się różnić (np. „Cluster 0” = prawdziwa grupa 2), bo algorytm nie zna „oficjalnych” nazw grup – to bez znaczenia.
K-Means **bezbłędnie** odkrył strukturę danych, choć nikt mu jej nie pokazał! 🎉

### Krok 10: Zapis modelu 💾
Model zapisujemy tak samo jak w regresji i klasyfikacji – później można go wczytać funkcją `load_model()`.

In [ ]:
# Zapis modelu (średnik ukrywa długi wydruk)
import os
os.makedirs('../modele', exist_ok=True)
save_model(model, '../modele/clustering_model');

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie: Znajdź grupy w nowych danych** 🔎

- **a)** Wygeneruj nowy zbiór: `make_blobs(n_samples=500, n_features=2, centers=4, cluster_std=0.8, random_state=1)`
  i zapisz go w tabeli `data2` z kolumnami `x` i `y`.
- **b)** Narysuj wykres rozrzutu `x` vs `y`. Ile grup widzisz gołym okiem?
- **c)** Uruchom `setup()` na `data2`, narysuj wykres łokcia. Jakie K wskazuje?
- **d)** Zbuduj K-Means z odpowiednią liczbą grup, przypisz grupy (`assign_model`) i narysuj wykres rozrzutu
  pokolorowany według kolumny `Cluster`.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `X2, y2 = make_blobs(...)`, a potem `pd.DataFrame(X2, columns=['x', 'y'])`.
- b) `sns.scatterplot(data=data2, x='x', y='y')`.
- c) `setup(data2, session_id=123)`, potem dowolny model `create_model('kmeans')` i `plot_model(..., plot='elbow')`.
- d) `create_model('kmeans', num_clusters=...)`, `assign_model(...)`, `sns.scatterplot(..., hue='Cluster')`.
  Mamy tylko 2 cechy, więc nie potrzebujemy PCA – rysujemy prosto na osiach `x` i `y`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

🌟 **Dla chętnych:** zwiększ `cluster_std` do 2.5 (grupy bardziej się nakładają) i sprawdź, co dzieje się z wykresem łokcia i sylwetką.

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Prawdziwe dane – czy K-Means odkryje gatunki irysów?
Dane syntetyczne były „grzeczne”. Sprawdźmy K-Means na prawdziwych danych, **ukrywając** przed nim gatunki.

- **a)** Wczytaj zbiór `iris`, usuń kolumnę `species` i uruchom `setup()` **z normalizacją** (`session_id=123`).
- **b)** Zbuduj K-Means z **3** grupami. Jaka jest sylwetka? Porównaj ją z wynikiem 0,80 dla danych syntetycznych.
- **c)** Przypisz grupy i porównaj je z prawdziwymi gatunkami w tabeli krzyżowej. Który gatunek K-Means „odkrył” bezbłędnie, a które pomieszał?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- W tym notebooku nie zaimportowaliśmy `get_data` – zrób to: `from pycaret.datasets import get_data`.
- Usuwanie kolumny: `iris.drop(columns='species')`.
- Normalizacja: `setup(..., normalize=True)` – pomiary są w różnych skalach.
- Tabela krzyżowa: `pd.crosstab(wyniki['Cluster'], iris['species'])` (kolejność wierszy jest taka sama, więc można porównywać).

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Gdy K-Means zawodzi – dwa księżyce i DBSCAN
K-Means zakłada, że grupy są „okrągłymi chmurkami” wokół środka. Co, jeśli mają inny kształt?

- **a)** Wygeneruj dane w kształcie **dwóch księżyców**: `make_moons(n_samples=500, noise=0.06, random_state=42)`
  (tabela `moons` z kolumnami `x`, `y`) i narysuj je.
- **b)** Uruchom `setup()` (`session_id=123`), zbuduj K-Means z **2** grupami i narysuj wynik (kolor = `Cluster`). Czy K-Means rozdzielił księżyce?
- **c)** Zbuduj model **DBSCAN** z parametrem `eps=0.25` i narysuj jego wynik. Który model poradził sobie lepiej?
- **d)** Porównaj **sylwetki** obu modeli. Czy wyższa sylwetka oznacza tu lepsze grupowanie? Co z tego wynika?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- `from sklearn.datasets import make_moons` – działa jak `make_blobs` (zwraca dane i prawdziwe etykiety).
- **DBSCAN** (nowy model, ID: `'dbscan'`) łączy w grupę punkty, które leżą **gęsto obok siebie** – „łańcuch” sąsiadów oddalonych
  o mniej niż `eps`. Nie trzeba podawać liczby grup! Wywołanie: `create_model('dbscan', eps=0.25)`.
- Wykres: `sns.scatterplot(data=wynik, x='x', y='y', hue='Cluster')`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- **Grupowanie** to uczenie nienadzorowane: szukamy grup podobnych obserwacji **bez etykiet**.
- **K-Means**: przypisz punkty do najbliższego środka → przesuń środki → powtarzaj.
- Liczbę grup K dobieramy **metodą łokcia**; jakość grupowania oceniamy **sylwetką** (bliżej 1 = lepiej).
- **PCA** pozwala narysować wielowymiarowe dane na płaskim wykresie.
- `assign_model()` dopisuje do danych kolumnę `Cluster`, a `groupby('Cluster').mean()` tworzy „portret” każdej grupy.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| grupowanie, analiza skupień | clustering | dzielenie danych na grupy podobnych obserwacji |
| skupienie, grupa | cluster | jedna grupa podobnych punktów |
| centroid, środek grupy | centroid | średni punkt grupy |
| metoda k-średnich | K-Means | algorytm grupowania wokół K środków |
| metoda łokcia | elbow method | wybór K w miejscu „zgięcia” krzywej |
| współczynnik sylwetki | silhouette coefficient | jak dobrze punkt pasuje do swojej grupy (−1…1) |
| analiza głównych składowych | Principal Component Analysis (PCA) | redukcja liczby wymiarów do wizualizacji |
| redukcja wymiarowości | dimensionality reduction | zmniejszanie liczby cech |
| segmentacja klientów | market / customer segmentation | podział klientów na grupy |
| dane syntetyczne | synthetic data | dane wygenerowane sztucznie |

➡️ **Za tydzień:** **wykrywanie anomalii** – poszukamy nietypowych klientów hurtowni.